# Historical collection of public stream metrics

This notebook documents the course project's Selenium collection from TwitchTracker. It uses the historical page selectors and stores the extracted stream history in the private collection directory. The collected table includes per-stream metrics and titles.

Dependencies are `selenium`, `pandas`, `openpyxl` and a local Firefox/WebDriver setup. Supply `TWITCHTRACKER_CHANNEL_LOGIN` in the local environment. `TWITCHTRACKER_FIREFOX_BINARY` is optional. Selenium uses its own browser profile for the session.

Running all cells defines the collection and export functions. The live call is commented out. This archived collection was kept separate from the course analysis rerun. Its historical selectors were preserved without checking the live site.


In [ ]:
import os
from pathlib import Path
import tempfile

repo_root = Path.cwd()
if repo_root.name == "collection" and repo_root.parent.name == "notebooks":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

private_data = Path(
    os.getenv("TWITCH_MODERATION_DATA") or repo_root / "data" / "private"
).expanduser()
private_collection = private_data / "collection"

import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By


In [ ]:
def read_stream_rows(driver):
    streams = driver.find_elements(By.ID, "streams")
    stream_info = streams[0].text.split("\n")[2:]
    stream_info = [item for item in stream_info if not item.startswith("+")]
    rows = []
    for i in range(0, len(stream_info), 6):
        if i + 5 >= len(stream_info):
            break
        rows.append({
            "Stream and Duration": stream_info[i],
            "Avg Viewers": stream_info[i + 1].replace(",", ""),
            "Max Viewers": stream_info[i + 2].replace(",", ""),
            "Followers": stream_info[i + 3].replace(",", ""),
            "Views": stream_info[i + 4].replace(",", ""),
            "Title": stream_info[i + 5],
        })
    return rows


def collect_stream_metrics(channel_login, last_page=74):
    if not re.fullmatch(r"[a-zA-Z0-9_]+", channel_login):
        raise ValueError("Supply a valid Twitch channel login in the environment.")

    options = webdriver.FirefoxOptions()
    firefox_binary = os.getenv("TWITCHTRACKER_FIREFOX_BINARY")
    if firefox_binary:
        options.binary_location = firefox_binary
    driver = webdriver.Firefox(options=options)
    try:
        driver.get(f"https://twitchtracker.com/{channel_login}/streams")
        rows = read_stream_rows(driver)
        for page in range(2, last_page + 1):
            next_page = driver.find_element(By.XPATH, f'//a[text()="{page}"]')
            next_page.click()
            rows.extend(read_stream_rows(driver))
    finally:
        driver.quit()
    return pd.DataFrame(rows)


In [ ]:
def export_stream_metrics():
    public_channel = os.environ["TWITCHTRACKER_CHANNEL_LOGIN"]
    metrics = collect_stream_metrics(public_channel)
    private_collection.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="stream-metrics-", dir=private_collection))
    metrics.to_excel(run_dir / "stream_metrics.xlsx", index=False)


# Run this explicitly once you have checked authorization and the site structure.
# export_stream_metrics()
